In [1]:
!pip install -q --no-deps raid-bench
!pip install -q transformers tokenizers sentencepiece tiktoken
!pip install -q --upgrade transformers  # get closest to 5.0.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 113.0 MB/s eta 0:00:00


In [2]:
import json

special_tokens = {
    "bos_token": "<s>",
    "eos_token": "</s>",
    "unk_token": "<unk>",
    "sep_token": "</s>",
    "pad_token": "<pad>",
    "cls_token": "<s>",
    "mask_token": "<mask>"
}

# Write to working dir, load from there
import shutil, os

WORK_MODEL_DIR = "/kaggle/working/model"
SRC = "/kaggle/input/models/mirceafinalboss/aigeneratedtextdetection/pytorch/default/1"

# Copy model to writable location
shutil.copytree(SRC, WORK_MODEL_DIR)

with open(f"{WORK_MODEL_DIR}/special_tokens_map.json", "w") as f:
    json.dump(special_tokens, f, indent=2)

print("Files now:", os.listdir(WORK_MODEL_DIR))

Files now: ['model.safetensors', 'config.json', 'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json', 'README.md', '.gitattributes']


In [3]:
import os
import json
import torch
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    matthews_corrcoef,
)

In [4]:
MODEL_DIR = "/kaggle/input/models/mirceafinalboss/aigeneratedtextdetection/pytorch/default/1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [5]:
@torch.no_grad()
def predict_batch(texts, model, tokenizer, device, batch_size=16, max_length=512):
    y_prob, y_pred = [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = (probs >= 0.5).long()
        y_prob.extend(probs.cpu().tolist())
        y_pred.extend(preds.cpu().tolist())
    return y_pred, y_prob

In [6]:
from raid.utils import load_data
from raid import run_detection

test_noadv_df = load_data(split="test", include_adversarial=False)
print(type(test_noadv_df))
print(test_noadv_df.head() if hasattr(test_noadv_df, "head") else test_noadv_df[:2])

100%|██████████| 81.0M/81.0M [00:00<00:00, 86.9MB/s]


<class 'pandas.core.frame.DataFrame'>
                                     id  \
0  64005577-3d63-4583-8945-7541d3e53e7d   
1  c2b9df67-4e29-45ca-bdcc-7065fb907b77   
2  07904f22-8530-4d3b-bf49-6bd1642d89f7   
3  dc5aa023-6f57-4f9c-833a-c0f322a994fa   
4  1b1ab19b-fe6f-458d-a666-06bbc1791534   

                                          generation  
0    The Sunspot Number, created by R.Wolf in 184...  
1    We present several analogies between convex ...  
2    Let H be a homology theory for algebraic var...  
3    The two parallel concepts of "small" sets of...  
4    We present new solutions to the strong explo...  


In [7]:
print(type(test_noadv_df))
print(test_noadv_df[:2] if isinstance(test_noadv_df, list) else test_noadv_df.head())

<class 'pandas.core.frame.DataFrame'>
                                     id  \
0  64005577-3d63-4583-8945-7541d3e53e7d   
1  c2b9df67-4e29-45ca-bdcc-7065fb907b77   
2  07904f22-8530-4d3b-bf49-6bd1642d89f7   
3  dc5aa023-6f57-4f9c-833a-c0f322a994fa   
4  1b1ab19b-fe6f-458d-a666-06bbc1791534   

                                          generation  
0    The Sunspot Number, created by R.Wolf in 184...  
1    We present several analogies between convex ...  
2    Let H be a homology theory for algebraic var...  
3    The two parallel concepts of "small" sets of...  
4    We present new solutions to the strong explo...  


In [8]:
def my_detector(texts: list[str]) -> list[float]:
    _, y_prob = predict_batch(texts, model, tokenizer, device)
    return y_prob

predictions = run_detection(my_detector, test_noadv_df)

In [9]:
y_prob = predictions
y_pred = [1 if p >= 0.5 else 0 for p in y_prob]
texts = test_noadv_df["generation"].tolist()
ids   = test_noadv_df["id"].tolist()

submission = [
    {"id": ids[i], "score": float(y_prob[i])}
    for i in range(len(ids))
]

output_path = "/kaggle/working/predictions.json"
with open(output_path, "w") as f:
    json.dump(submission, f, indent=2)

print(f"Saved {len(submission)} predictions → {output_path}")
print(os.listdir("/kaggle/working"))

TypeError: '>=' not supported between instances of 'dict' and 'float'